# SA Fine-tuning: vinai/phobert-base-v2

Phân loại cảm xúc **(NEGATIVE / NEUTRAL / POSITIVE)** theo từng aspect và tổng thể.

**Dữ liệu:** ViSD4SA — format `<span> : <full_text>` (ABSA pair approach).

In [1]:
!pip install -q jsonlines transformers datasets py_vncorenlp scikit-learn accelerate

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 33.1 MB/s eta 0:00:00


## 1. Cấu hình

In [2]:
from os import path

# ── Đổi đường dẫn tại đây nếu dùng Colab ──────────────────────────────────
ROOT = "/kaggle/working"
# ──────────────────────────────────────────────────────────────────────────

DATASET       = "/kaggle/input/datasets/lchhong/uit-visd4sa"
MODEL_SAVE    = "/kaggle/working/models/NER"
VNCORENLP_DIR = ROOT
BASE_MODEL    = "vinai/phobert-base-v2"

EPOCHS       = 5
LR           = 2e-5
BATCH_SIZE   = 64
MAX_SEQ_LEN  = 256
WARMUP_STEPS = 200
WEIGHT_DECAY = 0.01

SENTIMENT_LABELS = ["NEGATIVE", "NEUTRAL", "POSITIVE"]
label2id = {l: i for i, l in enumerate(SENTIMENT_LABELS)}
id2label = {i: l for i, l in enumerate(SENTIMENT_LABELS)}

## 2. VnCoreNLP — Tách từ tiếng Việt

In [3]:
import py_vncorenlp

py_vncorenlp.download_model(save_dir=VNCORENLP_DIR)
segmenter = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=VNCORENLP_DIR)

--2026-03-05 03:57:14--  https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master/VnCoreNLP-1.2.jar
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 27412703 (26M) [application/octet-stream]
Saving to: ‘VnCoreNLP-1.2.jar’

     0K .......... .......... .......... .......... ..........  0% 7.07M 4s
    50K .......... .......... .......... .......... ..........  0% 22.6M 2s
   100K .......... .......... .......... .......... ..........  0% 9.86M 2s
   150K .......... .......... .......... .......... ..........  0% 43.1M 2s
   200K .......... .......... .......... .......... ..........  0% 50.7M 2s
   250K .......... .......... .......... .......... ..........  1% 11.6M 2s
   300K .......... .......... .......... .......... ..........  1% 54.7M 2s
   350K ..

2026-03-05 03:57:16 INFO  WordSegmenter:24 - Loading Word Segmentation model


## 3. Tải dữ liệu

In [4]:
import jsonlines

def load_jsonl(fp):
    with jsonlines.open(fp) as f:
        return list(f.iter())

train_raw = load_jsonl(path.join(DATASET, "train.jsonl"))
dev_raw   = load_jsonl(path.join(DATASET, "dev.jsonl"))
test_raw  = load_jsonl(path.join(DATASET, "test.jsonl"))

print(f"Train: {len(train_raw)} | Dev: {len(dev_raw)} | Test: {len(test_raw)}")

Train: 7785 | Dev: 1112 | Test: 2225


## 4. Tiền xử lý

In [5]:
import unicodedata
import re

def clean_text(text):
    chars = [c if (unicodedata.category(c)[0] in ('L', 'N') or c == ' ') else ' ' for c in text]
    return re.sub(r' +', ' ', ''.join(chars)).strip().lower()


def word_segment(text):
    segs = segmenter.word_segment(text)
    return segs[0] if segs else text


def preprocess(text):
    return word_segment(clean_text(text))

## 5. Xây dựng tập dữ liệu SA

Mỗi review tạo ra:
- 1 row tổng thể: `(full_text, overall_sentiment)`
- N rows theo aspect: `(span : full_text, aspect_sentiment)`

In [6]:
from collections import Counter


def overall_sentiment(labels):
    """Majority voting. Tie (P == N) → NEUTRAL."""
    counts = Counter(lbl.split("#")[1] for lbl in labels)
    if counts["POSITIVE"] == counts["NEGATIVE"]:
        return "NEUTRAL"
    return counts.most_common(1)[0][0]


def build_sa_rows(raw_data):
    rows = []
    for item in raw_data:
        text_seg = preprocess(item["text"])
        labels   = [lbl for _, _, lbl in item["labels"]]

        # Row tổng thể
        rows.append({"text": text_seg, "labels": label2id[overall_sentiment(labels)]})

        # Row theo từng aspect (span : full_text)
        seen = {}
        for start, end, label in item["labels"]:
            aspect, sentiment = label.split("#")
            if aspect not in seen:
                seen[aspect] = True
                span_seg  = preprocess(item["text"][start:end])
                pair_text = f"{span_seg} : {text_seg}"
                rows.append({"text": pair_text, "labels": label2id[sentiment]})

    return rows


train_rows = build_sa_rows(train_raw) + build_sa_rows(dev_raw)
test_rows  = build_sa_rows(test_raw)

print(f"Train+Dev: {len(train_rows):,} samples | Test: {len(test_rows):,} samples")

Train+Dev: 34,151 samples | Test: 8,483 samples


## 6. DataFrame + Class Weights

In [7]:
import pandas as pd
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

train_df = pd.DataFrame(train_rows)
test_df  = pd.DataFrame(test_rows)
train_df["labels"] = train_df["labels"].astype(int)
test_df["labels"]  = test_df["labels"].astype(int)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(SENTIMENT_LABELS)),
    y=train_df["labels"],
)
print("Class weights:", dict(zip(SENTIMENT_LABELS, class_weights.round(4))))
print(f"\nPhân phối nhãn:\n{train_df['labels'].value_counts().rename(id2label)}")

Class weights: {'NEGATIVE': np.float64(1.0275), 'NEUTRAL': np.float64(3.972), 'POSITIVE': np.float64(0.5634)}

Phân phối nhãn:
labels
POSITIVE    20206
NEGATIVE    11079
NEUTRAL      2866
Name: count, dtype: int64


## 7. Fine-tune PhoBERT

In [8]:
import torch
import torch.nn as nn
import logging
import numpy as np
from sklearn.metrics import f1_score, classification_report as sk_report
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

logging.getLogger("transformers").setLevel(logging.WARNING)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)


class SADataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            max_length=MAX_SEQ_LEN,
            padding=False,
        )
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item


class WeightedTrainer(Trainer):
    """Trainer tùy chỉnh: thêm class weights vào CrossEntropyLoss."""
    def __init__(self, class_weights, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = torch.tensor(class_weights, dtype=torch.float)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        weight  = self.class_weights.to(outputs.logits.device)
        loss    = nn.CrossEntropyLoss(weight=weight)(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss


def compute_metrics(p):
    preds  = np.argmax(p.predictions, axis=1)
    labels = p.label_ids
    return {"f1": f1_score(labels, preds, average="macro")}


train_ds = SADataset(train_df["text"].tolist(), train_df["labels"].tolist())
test_ds  = SADataset(test_df["text"].tolist(),  test_df["labels"].tolist())

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(SENTIMENT_LABELS),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

collator = DataCollatorWithPadding(tokenizer)

training_args = TrainingArguments(
    output_dir=MODEL_SAVE,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,
    logging_steps=50,
    report_to="none",
)

trainer = WeightedTrainer(
    class_weights=class_weights,
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()

config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/540M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=2670, training_loss=0.28792294741569835, metrics={'train_runtime': 2066.3757, 'train_samples_per_second': 82.635, 'train_steps_per_second': 1.292, 'total_flos': 1.1694556855137744e+16, 'train_loss': 0.28792294741569835, 'epoch': 5.0})

## 8. Đánh giá

In [9]:
preds_output = trainer.predict(test_ds)
preds  = np.argmax(preds_output.predictions, axis=1)
labels = preds_output.label_ids

print(sk_report(labels, preds, target_names=SENTIMENT_LABELS, digits=4))

              precision    recall  f1-score   support

    NEGATIVE     0.9498    0.9125    0.9307      2776
     NEUTRAL     0.5675    0.6944    0.6246       648
    POSITIVE     0.9618    0.9549    0.9583      5059

    accuracy                         0.9211      8483
   macro avg     0.8263    0.8539    0.8379      8483
weighted avg     0.9277    0.9211    0.9238      8483



## 9. Lưu mô hình

In [10]:
import os

os.makedirs(MODEL_SAVE, exist_ok=True)
model.save_pretrained(MODEL_SAVE)
tokenizer.save_pretrained(MODEL_SAVE)
print(f"Model saved -> {MODEL_SAVE}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved -> /kaggle/working/models/NER
